# Notebook d'analyse — IA Data Scientist

Ce notebook a été généré automatiquement à la fin du pipeline. Il permet de **rejouer les calculs**, **vérifier les résultats** et **continuer le travail** (nouvelles features, autres modèles, réglages différents), sans dépendre du code interne du projet.

- **Variable cible :** `Unnamed: 0`
- **Type de problème :** `regression`
- **Modèle champion :** `XGBoost`
- **Score de validation croisée :** `79.0825`


## 1. Imports et chargement des données

Les fichiers ci-dessous ont été exportés automatiquement par le pipeline dans `data/processed/`. Ils sont indépendants du code source du projet : n'importe quel outil (Python, R, Excel) peut les ouvrir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Chemins relatifs depuis reports/final/
CHEMIN_DONNEES = "../../data/processed"
CHEMIN_MODELE = "../../model"

X_train = pd.read_csv(f"{CHEMIN_DONNEES}/X_train.csv")
X_test = pd.read_csv(f"{CHEMIN_DONNEES}/X_test.csv")
y_train = pd.read_csv(f"{CHEMIN_DONNEES}/y_train.csv").iloc[:, 0]
y_test = pd.read_csv(f"{CHEMIN_DONNEES}/y_test.csv").iloc[:, 0]

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

## 2. Statistiques descriptives

Ces calculs (moyenne, somme, écart-type, valeurs manquantes) sont ceux qu'un data scientist ferait manuellement en début d'analyse. Ils sont recalculés ici directement sur les données réelles.

In [ ]:
# Statistiques générales sur les features (X_train)
X_train.describe().T

In [ ]:
# Moyenne, somme et écart-type par colonne numérique
stats = pd.DataFrame({
    "moyenne": X_train.mean(numeric_only=True),
    "somme": X_train.sum(numeric_only=True),
    "ecart_type": X_train.std(numeric_only=True),
    "min": X_train.min(numeric_only=True),
    "max": X_train.max(numeric_only=True),
})
stats

In [ ]:
# Valeurs manquantes par colonne
X_train.isna().sum().sort_values(ascending=False)

## 3. Distribution de la variable cible

In [ ]:
print(y_train.describe())

plt.figure(figsize=(6, 3.5))
plt.hist(y_train, bins=30, color="#2E86AB")
plt.title(f"Distribution de la cible : Unnamed: 0")
plt.tight_layout()
plt.show()

## 4. Modèle champion

Modèle retenu par l'AutoML : **XGBoost**

Le modèle ci-dessous est celui déjà entraîné par le pipeline (chargé depuis le fichier `.joblib`), avec les hyperparamètres suivants :

- `n_estimators` = `250`
- `max_depth` = `9`
- `learning_rate` = `0.26919152711343286`
- `subsample` = `0.6693581826684941`
- `colsample_bytree` = `0.6360430105797488`

In [ ]:
modele = joblib.load(f"{CHEMIN_MODELE}/model_champion.joblib")
modele

## 5. Évaluation détaillée

Cette section recalcule les prédictions du modèle sur le jeu de test et affiche le détail (matrice de confusion, rapport de classification, ou métriques de régression selon le type de problème).

In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

predictions = modele.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, predictions, alpha=0.5, color="#2E86AB")
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--"
)
plt.xlabel("Valeur réelle")
plt.ylabel("Valeur prédite")
plt.title("Prédit vs Réel")
plt.tight_layout()
plt.show()

## 6. Importance des variables

Disponible si le modèle expose l'attribut `feature_importances_` (Random Forest, XGBoost, LightGBM, CatBoost, ...).

In [ ]:
if hasattr(modele, "feature_importances_"):
    importances = pd.Series(
        modele.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)

    importances.head(20).plot(
        kind="barh", figsize=(7, 6), color="#2E86AB"
    )
    plt.title("Top 20 variables les plus importantes")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Ce modèle n'expose pas feature_importances_.")

## 7. Pour aller plus loin

Quelques pistes pour continuer ce travail directement dans ce notebook :

- Essayer d'autres hyperparamètres avec `modele.set_params(...)` puis `modele.fit(X_train, y_train_encode)`
- Ajouter de nouvelles colonnes calculées à `X_train` / `X_test` (feature engineering manuel)
- Comparer avec un autre algorithme (`RandomForestClassifier`, `LGBMClassifier`, ...)
- Exporter `X_train` / `X_test` en `.csv` ou `.rds` pour poursuivre l'analyse sous R
- Utiliser `shap` pour une explicabilité plus fine que l'importance de variables classique

In [ ]:
# Espace libre pour continuer l'analyse
